In [7]:
pip install pandas numpy matplotlib seaborn openpyxl scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.9 MB 5.6 MB/s eta 0:00:02
   ------- -------------------------------- 1.6/8.9 MB 9.4 MB/s eta 0:00:01
   ----------- ---------------------------- 2.6/8.9 MB 4.9 MB/s eta 0:00:02
   -------------- ------------------------- 3.1/8.9 MB 4.2 MB/s eta 0:00:02
   ----------------- ---------------------- 3.9/8.9 MB 4.2 MB/s eta 0:00:02
   -------------------- ------------------- 4.5/8.9 MB 4.1 MB/s eta 0:00:02
   ------------------------ --------------- 5.5/8.9 MB 4.1 MB/s eta 0:00:01
   ---------------------------- ----------- 6.3/8.9 MB 4.1 MB/s eta 0:00:01
   ------------------------------- -------- 7.1/8.9 MB 4.0 MB/s eta 0:00:01
   ----------------------------------- ---- 7.9/8.9 MB 4.0 MB/s eta 0:00:01
   ------------------------------------

In [8]:
"""
=============================================================
TELECOM ALARM DATASET — ML PREPROCESSING PIPELINE
=============================================================
Produces:
  ml_dataset.csv         — full feature + label dataset
  X_train.csv / X_test.csv
  y_train.csv / y_test.csv
  preprocessing_meta.json — scaler params, encodings, col lists
  preprocessing_report.txt — full report of every decision made

Steps:
  3a — Data Cleaning        (outliers, REG_OTHER, floor fix)
  3b — Feature Engineering  (temporal, hourly aggregations, encoding)
  3c — Target Label Gen     (4 thresholds per row per start_hour)
  3d — Feature Scaling      (log transform + StandardScaler)
  3e — Train/Test Split     (time-aware, 7 train / 3 test days)
=============================================================
"""

import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder

OUTPUT_DIR = './ml_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

REPORT_LINES = []
def log(msg=''):
    print(msg)
    REPORT_LINES.append(msg)

# ── Constants ─────────────────────────────────────────────────
FLOOR_INACTIVE_SECONDS = 300      # 5 min — minimum meaningful inactive time
FLOOR_INACTIVE_MINUTES = 5.0      # floor for derived threshold labels
FLOOR_FLUCTUATION      = 1        # floor for fluctuation threshold labels
INACTIVE_CAP_PERCENTILE = 99      # cap outliers at this percentile
FLUCTUATION_CAP        = 7        # from EDA: P99 fluctuation = 7
MIN_WINDOW_HOURS       = 2
MAX_WINDOW_HOURS       = 8

# Sensitivity factors per hour block (from EDA findings)
def sensitivity_factor(start_hour_0indexed):
    h = start_hour_0indexed  # 0 = midnight, 23 = 11pm
    if 17 <= h <= 22:        # 18:00–23:00 — evening, strict
        return 0.5
    elif 6 <= h <= 10:       # 07:00–11:00 — morning peak, normal
        return 1.0
    else:                    # all other hours — off-peak, relaxed
        return 1.25

log("=" * 62)
log("TELECOM ALARM ML PREPROCESSING PIPELINE")
log("=" * 62)

# =============================================================
# LOAD
# =============================================================
log("\n[LOAD] Reading anonymized dataset...")
df = pd.read_csv('anonymized_alarm_data.csv')
df['Alarm Raised Date'] = pd.to_datetime(df['Alarm Raised Date'])
df = df.sort_values('Alarm Raised Date').reset_index(drop=True)

inactive_cols = [f'Hour_{h}_Inactive' for h in range(1, 25)]
fluct_cols    = [f'Hour_{h}_Fluctuation' for h in range(1, 25)]

log(f"  Loaded: {df.shape[0]} rows × {df.shape[1]} cols")
log(f"  Dates:  {df['Alarm Raised Date'].min().date()} → {df['Alarm Raised Date'].max().date()}")
log(f"  Vendors: {sorted(df['Vendor'].unique().tolist())}")

# =============================================================
# STEP 3a — DATA CLEANING
# =============================================================
log("\n" + "=" * 62)
log("STEP 3a — DATA CLEANING")
log("=" * 62)

original_len = len(df)

# ── 3a-1: Remove REG_OTHER (hash-fallback anomalous cells) ───
reg_other_mask = df['Cell'].str.startswith('CELL_')
log(f"\n  [3a-1] REG_OTHER (anomalous hash cells): {reg_other_mask.sum()} rows → removed")
df = df[~reg_other_mask].reset_index(drop=True)

# ── 3a-2: Cap inactive time outliers at P99 ──────────────────
inactive_cap = np.percentile(df['Total_Inactive_Time_Seconds'], INACTIVE_CAP_PERCENTILE)
rows_capped = (df['Total_Inactive_Time_Seconds'] > inactive_cap).sum()
df['Total_Inactive_Time_Seconds'] = df['Total_Inactive_Time_Seconds'].clip(upper=inactive_cap)
for col in inactive_cols:
    df[col] = df[col].clip(upper=inactive_cap)
log(f"  [3a-2] Inactive time capped at P99 = {inactive_cap:.0f}s ({inactive_cap/60:.1f} min)")
log(f"         Rows affected: {rows_capped}")

# ── 3a-3: Cap fluctuation at P99 ─────────────────────────────
rows_fluct_capped = (df['Fluctuation_Count'] > FLUCTUATION_CAP).sum()
df['Fluctuation_Count'] = df['Fluctuation_Count'].clip(upper=FLUCTUATION_CAP)
for col in fluct_cols:
    df[col] = df[col].clip(upper=FLUCTUATION_CAP)
log(f"  [3a-3] Fluctuation capped at {FLUCTUATION_CAP} (P99 value)")
log(f"         Rows affected: {rows_fluct_capped}")

# ── 3a-4: Recalculate Total after capping ────────────────────
df['Total_Inactive_Time_Seconds'] = df[inactive_cols].sum(axis=1)
df['Fluctuation_Count']           = df[fluct_cols].sum(axis=1)
log(f"  [3a-4] Totals recalculated from hourly values after capping")

log(f"\n  Rows after cleaning: {len(df)} (removed {original_len - len(df)})")

# =============================================================
# STEP 3b — FEATURE ENGINEERING
# =============================================================
log("\n" + "=" * 62)
log("STEP 3b — FEATURE ENGINEERING")
log("=" * 62)

# ── 3b-1: Extract region from cell name ──────────────────────
df['Region'] = df['Cell'].apply(
    lambda x: '_'.join(x.split('_')[:2]) if len(x.split('_')) >= 2 else 'UNKNOWN'
)

# ── 3b-2: Temporal features ──────────────────────────────────
log("\n  [3b-2] Temporal features:")
df['DayOfWeek']    = df['Alarm Raised Date'].dt.dayofweek       # 0=Mon, 6=Sun
df['DayName']      = df['Alarm Raised Date'].dt.day_name()
df['IsWeekend']    = (df['DayOfWeek'] >= 5).astype(int)
df['Month']        = df['Alarm Raised Date'].dt.month
df['DayOfMonth']   = df['Alarm Raised Date'].dt.day
df['DayOfYear']    = df['Alarm Raised Date'].dt.dayofyear

# Cyclical encoding for day of week (preserves Mon≈Sun proximity)
df['DayOfWeek_sin'] = np.sin(2 * np.pi * df['DayOfWeek'] / 7)
df['DayOfWeek_cos'] = np.cos(2 * np.pi * df['DayOfWeek'] / 7)

log(f"    DayOfWeek, IsWeekend, Month, DayOfMonth, DayOfYear")
log(f"    DayOfWeek_sin, DayOfWeek_cos (cyclical encoding)")

# ── 3b-3: Hourly aggregation features ────────────────────────
log("\n  [3b-3] Hourly aggregation features:")

# Time block aggregations (based on EDA findings)
# Block 1: Night       H1–H6   (off-peak, early morning)
# Block 2: Morning     H7–H12  (peak activity)
# Block 3: Afternoon   H13–H18 (dead zone)
# Block 4: Evening     H19–H24 (late activity surge)

blocks = {
    'Night':     list(range(1, 7)),
    'Morning':   list(range(7, 13)),
    'Afternoon': list(range(13, 19)),
    'Evening':   list(range(19, 25)),
}

for block_name, hours in blocks.items():
    i_cols = [f'Hour_{h}_Inactive' for h in hours]
    f_cols = [f'Hour_{h}_Fluctuation' for h in hours]

    df[f'{block_name}_Inactive_Sum']  = df[i_cols].sum(axis=1)
    df[f'{block_name}_Inactive_Max']  = df[i_cols].max(axis=1)
    df[f'{block_name}_Inactive_Mean'] = df[i_cols].mean(axis=1)
    df[f'{block_name}_Fluct_Sum']     = df[f_cols].sum(axis=1)
    df[f'{block_name}_Fluct_Max']     = df[f_cols].max(axis=1)
    df[f'{block_name}_Active_Hours']  = (df[i_cols] > FLOOR_INACTIVE_SECONDS).sum(axis=1)

log(f"    4 time blocks × 6 features = 24 block features")

# Peak hour features (from EDA: H9 and H24 are peaks)
df['PeakMorning_H9_Inactive']  = df['Hour_9_Inactive']
df['PeakNight_H24_Inactive']   = df['Hour_24_Inactive']
df['PeakMorning_H10_Fluct']    = df['Hour_10_Fluctuation']
df['Total_Active_Hours']        = (df[inactive_cols] > FLOOR_INACTIVE_SECONDS).sum(axis=1)
df['Active_Hour_Ratio']         = df['Total_Active_Hours'] / 24

log(f"    Peak hour features: H9 inactive, H24 inactive, H10 fluctuation")
log(f"    Total_Active_Hours, Active_Hour_Ratio")

# Inactive time concentration (Gini-like measure)
# High = inactive concentrated in few hours, Low = spread evenly
def concentration_score(row_vals):
    total = row_vals.sum()
    if total == 0:
        return 0.0
    fracs = np.sort(row_vals / total)[::-1]
    return fracs[0]  # fraction held by the most active hour

df['Inactive_Concentration'] = df[inactive_cols].apply(
    lambda row: concentration_score(row.values), axis=1
)
df['Fluct_Concentration'] = df[fluct_cols].apply(
    lambda row: concentration_score(row.values), axis=1
)
log(f"    Inactive_Concentration, Fluct_Concentration (peak-hour dominance ratio)")

# Log transforms of total inactive (from EDA: right-skewed)
df['Log_Total_Inactive'] = np.log1p(df['Total_Inactive_Time_Seconds'])
df['Log_Fluctuation']    = np.log1p(df['Fluctuation_Count'])
log(f"    Log_Total_Inactive, Log_Fluctuation (log1p transforms)")

# ── 3b-4: Rolling features (lag from previous days) ──────────
log("\n  [3b-4] Rolling/lag features (day-level):")

# Sort by date for rolling
# For each date, compute rolling mean of last available days
# NOTE: with only 10 days this is limited but still informative

date_level = df.groupby('Alarm Raised Date').agg(
    Daily_Mean_Inactive  = ('Total_Inactive_Time_Seconds', 'mean'),
    Daily_Mean_Fluct     = ('Fluctuation_Count', 'mean'),
    Daily_Active_Hours   = ('Total_Active_Hours', 'mean'),
).reset_index().sort_values('Alarm Raised Date')

# Lag 1: previous available day stats
date_level['Lag1_Mean_Inactive'] = date_level['Daily_Mean_Inactive'].shift(1)
date_level['Lag1_Mean_Fluct']    = date_level['Daily_Mean_Fluct'].shift(1)

# Rolling mean of all previous days (expanding)
date_level['Rolling_Mean_Inactive'] = date_level['Daily_Mean_Inactive'].expanding().mean().shift(1)
date_level['Rolling_Mean_Fluct']    = date_level['Daily_Mean_Fluct'].expanding().mean().shift(1)

# Fill NaN for first day with same-day values (no prior history)
date_level['Lag1_Mean_Inactive'].fillna(date_level['Daily_Mean_Inactive'], inplace=True)
date_level['Lag1_Mean_Fluct'].fillna(date_level['Daily_Mean_Fluct'], inplace=True)
date_level['Rolling_Mean_Inactive'].fillna(date_level['Daily_Mean_Inactive'], inplace=True)
date_level['Rolling_Mean_Fluct'].fillna(date_level['Daily_Mean_Fluct'], inplace=True)

# Merge back to main df
df = df.merge(
    date_level[['Alarm Raised Date', 'Lag1_Mean_Inactive', 'Lag1_Mean_Fluct',
                'Rolling_Mean_Inactive', 'Rolling_Mean_Fluct']],
    on='Alarm Raised Date', how='left'
)
log(f"    Lag1_Mean_Inactive, Lag1_Mean_Fluct")
log(f"    Rolling_Mean_Inactive, Rolling_Mean_Fluct")

# ── 3b-5: Vendor & Region encoding ───────────────────────────
log("\n  [3b-5] Vendor & Region encoding:")

# Vendor: one-hot (only 3 vendors, low cardinality)
vendor_dummies = pd.get_dummies(df['Vendor'], prefix='Vendor')
df = pd.concat([df, vendor_dummies], axis=1)
log(f"    Vendor: one-hot → {list(vendor_dummies.columns)}")

# Region: one-hot (9 regions, manageable)
region_dummies = pd.get_dummies(df['Region'], prefix='Region')
df = pd.concat([df, region_dummies], axis=1)
log(f"    Region: one-hot → {len(region_dummies.columns)} columns")

log(f"\n  Total features after engineering: ~{len(df.columns)}")

# =============================================================
# STEP 3c — TARGET LABEL GENERATION
# =============================================================
log("\n" + "=" * 62)
log("STEP 3c — TARGET LABEL GENERATION")
log("=" * 62)
log("""
  Strategy:
    For each row (cell × date), we generate labels for every
    possible start_hour (0–23). Each row × start_hour combination
    becomes one training sample.

    threshold_hours:
      = count of hours in next MAX_WINDOW_HOURS with inactive
        > FLOOR_INACTIVE_SECONDS, clipped to [MIN, MAX]
      → tells model: "this many hours are worth auditing"

    threshold_inactive_time (minutes):
      = P75 of inactive time across window hours × sensitivity
      floor at FLOOR_INACTIVE_MINUTES

    threshold_fluctuation:
      = P90 of total fluctuation across window hours
      floor at FLOOR_FLUCTUATION

    each_hour_fluctuation:
      = P90 of per-hour max fluctuation across window hours
      floor at FLOOR_FLUCTUATION
""")

rows = []

for idx, row in df.iterrows():
    base_features = row.copy()

    for start_h in range(24):  # 0 = midnight, 23 = 11pm

        # Hours in the maximum possible window (next 8 hours)
        max_window = [(start_h + i) % 24 + 1 for i in range(MAX_WINDOW_HOURS)]

        # ── threshold_hours ───────────────────────────────────
        active_in_window = sum(
            1 for h in max_window
            if row[f'Hour_{h}_Inactive'] > FLOOR_INACTIVE_SECONDS
        )
        t_hours = int(np.clip(active_in_window + 1, MIN_WINDOW_HOURS, MAX_WINDOW_HOURS))

        # ── Use the predicted window (not max) for other labels ─
        actual_window = [(start_h + i) % 24 + 1 for i in range(t_hours)]

        window_inactive = np.array([row[f'Hour_{h}_Inactive'] for h in actual_window])
        window_fluct    = np.array([row[f'Hour_{h}_Fluctuation'] for h in actual_window])

        # ── threshold_inactive_time ───────────────────────────
        sf = sensitivity_factor(start_h)
        p75_inactive_sec = np.percentile(window_inactive, 75)
        t_inactive_min = max(
            FLOOR_INACTIVE_MINUTES,
            round((p75_inactive_sec * sf) / 60, 2)
        )

        # ── threshold_fluctuation ─────────────────────────────
        p90_fluct = np.percentile(window_fluct, 90)
        t_fluct = max(FLOOR_FLUCTUATION, round(p90_fluct, 1))

        # ── each_hour_fluctuation ─────────────────────────────
        per_hour_max_fluct = np.array([
            row[f'Hour_{h}_Fluctuation'] for h in actual_window
        ])
        p90_hourly = np.percentile(per_hour_max_fluct, 90)
        t_each_hour = max(FLOOR_FLUCTUATION, round(p90_hourly, 1))

        # ── Start hour features (cyclical) ────────────────────
        start_hour_sin = np.sin(2 * np.pi * start_h / 24)
        start_hour_cos = np.cos(2 * np.pi * start_h / 24)

        # ── Window inactive/fluctuation summary features ──────
        window_inactive_sum  = window_inactive.sum()
        window_inactive_mean = window_inactive.mean()
        window_fluct_sum     = window_fluct.sum()

        sample = {
            # identifiers
            'Cell':             row['Cell'],
            'Alarm_Raised_Date': row['Alarm Raised Date'],
            'Start_Hour':       start_h + 1,  # 1-indexed for readability

            # start hour context features
            'Start_Hour_sin':   start_hour_sin,
            'Start_Hour_cos':   start_hour_cos,
            'IsEvening':        int(17 <= start_h <= 22),
            'IsMorningPeak':    int(6 <= start_h <= 10),
            'IsOffPeak':        int(not (17 <= start_h <= 22) and not (6 <= start_h <= 10)),
            'Sensitivity_Factor': sf,

            # window summary features
            'Window_Inactive_Sum':  window_inactive_sum,
            'Window_Inactive_Mean': window_inactive_mean,
            'Window_Fluct_Sum':     window_fluct_sum,
            'Window_Active_Hours':  active_in_window,

            # targets
            'TARGET_threshold_hours':         t_hours,
            'TARGET_threshold_inactive_min':  t_inactive_min,
            'TARGET_threshold_fluctuation':   t_fluct,
            'TARGET_each_hour_fluctuation':   t_each_hour,

            # all engineered base features
            'DayOfWeek':          row['DayOfWeek'],
            'IsWeekend':          row['IsWeekend'],
            'Month':              row['Month'],
            'DayOfMonth':         row['DayOfMonth'],
            'DayOfYear':          row['DayOfYear'],
            'DayOfWeek_sin':      row['DayOfWeek_sin'],
            'DayOfWeek_cos':      row['DayOfWeek_cos'],
            'Total_Inactive_Time_Seconds': row['Total_Inactive_Time_Seconds'],
            'Fluctuation_Count':  row['Fluctuation_Count'],
            'Log_Total_Inactive': row['Log_Total_Inactive'],
            'Log_Fluctuation':    row['Log_Fluctuation'],
            'Total_Active_Hours': row['Total_Active_Hours'],
            'Active_Hour_Ratio':  row['Active_Hour_Ratio'],
            'Inactive_Concentration': row['Inactive_Concentration'],
            'Fluct_Concentration':    row['Fluct_Concentration'],
            'PeakMorning_H9_Inactive': row['PeakMorning_H9_Inactive'],
            'PeakNight_H24_Inactive':  row['PeakNight_H24_Inactive'],
            'PeakMorning_H10_Fluct':   row['PeakMorning_H10_Fluct'],
            'Night_Inactive_Sum':    row['Night_Inactive_Sum'],
            'Night_Inactive_Max':    row['Night_Inactive_Max'],
            'Night_Inactive_Mean':   row['Night_Inactive_Mean'],
            'Night_Fluct_Sum':       row['Night_Fluct_Sum'],
            'Night_Fluct_Max':       row['Night_Fluct_Max'],
            'Night_Active_Hours':    row['Night_Active_Hours'],
            'Morning_Inactive_Sum':  row['Morning_Inactive_Sum'],
            'Morning_Inactive_Max':  row['Morning_Inactive_Max'],
            'Morning_Inactive_Mean': row['Morning_Inactive_Mean'],
            'Morning_Fluct_Sum':     row['Morning_Fluct_Sum'],
            'Morning_Fluct_Max':     row['Morning_Fluct_Max'],
            'Morning_Active_Hours':  row['Morning_Active_Hours'],
            'Afternoon_Inactive_Sum':  row['Afternoon_Inactive_Sum'],
            'Afternoon_Inactive_Max':  row['Afternoon_Inactive_Max'],
            'Afternoon_Inactive_Mean': row['Afternoon_Inactive_Mean'],
            'Afternoon_Fluct_Sum':     row['Afternoon_Fluct_Sum'],
            'Afternoon_Fluct_Max':     row['Afternoon_Fluct_Max'],
            'Afternoon_Active_Hours':  row['Afternoon_Active_Hours'],
            'Evening_Inactive_Sum':  row['Evening_Inactive_Sum'],
            'Evening_Inactive_Max':  row['Evening_Inactive_Max'],
            'Evening_Inactive_Mean': row['Evening_Inactive_Mean'],
            'Evening_Fluct_Sum':     row['Evening_Fluct_Sum'],
            'Evening_Fluct_Max':     row['Evening_Fluct_Max'],
            'Evening_Active_Hours':  row['Evening_Active_Hours'],
            'Lag1_Mean_Inactive':    row['Lag1_Mean_Inactive'],
            'Lag1_Mean_Fluct':       row['Lag1_Mean_Fluct'],
            'Rolling_Mean_Inactive': row['Rolling_Mean_Inactive'],
            'Rolling_Mean_Fluct':    row['Rolling_Mean_Fluct'],
        }

        # Add vendor and region one-hot columns
        for col in df.columns:
            if col.startswith('Vendor_') or col.startswith('Region_'):
                sample[col] = row[col]

        # Add all 48 hourly raw features
        for h in range(1, 25):
            sample[f'Hour_{h}_Inactive']    = row[f'Hour_{h}_Inactive']
            sample[f'Hour_{h}_Fluctuation'] = row[f'Hour_{h}_Fluctuation']

        rows.append(sample)

    if idx % 1000 == 0:
        print(f"  Processing row {idx}/{len(df)}...", end='\r')

ml_df = pd.DataFrame(rows)
log(f"\n  ML dataset shape: {ml_df.shape}")
log(f"  (14,046 rows × 24 start_hours = {len(ml_df):,} training samples)")

# =============================================================
# STEP 3d — FEATURE SCALING
# =============================================================
log("\n" + "=" * 62)
log("STEP 3d — FEATURE SCALING")
log("=" * 62)

# Define feature groups
TARGET_COLS = [
    'TARGET_threshold_hours',
    'TARGET_threshold_inactive_min',
    'TARGET_threshold_fluctuation',
    'TARGET_each_hour_fluctuation',
]

ID_COLS = ['Cell', 'Alarm_Raised_Date', 'Start_Hour']

# All feature columns (everything except targets and IDs)
FEATURE_COLS = [c for c in ml_df.columns if c not in TARGET_COLS + ID_COLS]

log(f"\n  Feature columns: {len(FEATURE_COLS)}")
log(f"  Target columns:  {len(TARGET_COLS)}")

# Columns to scale with StandardScaler (continuous numeric)
SCALE_COLS = [
    'Total_Inactive_Time_Seconds', 'Log_Total_Inactive',
    'Fluctuation_Count', 'Log_Fluctuation',
    'Total_Active_Hours', 'Active_Hour_Ratio',
    'Inactive_Concentration', 'Fluct_Concentration',
    'PeakMorning_H9_Inactive', 'PeakNight_H24_Inactive',
    'PeakMorning_H10_Fluct',
    'Night_Inactive_Sum', 'Night_Inactive_Max', 'Night_Inactive_Mean',
    'Night_Fluct_Sum', 'Night_Fluct_Max',
    'Morning_Inactive_Sum', 'Morning_Inactive_Max', 'Morning_Inactive_Mean',
    'Morning_Fluct_Sum', 'Morning_Fluct_Max',
    'Afternoon_Inactive_Sum', 'Afternoon_Inactive_Max', 'Afternoon_Inactive_Mean',
    'Afternoon_Fluct_Sum', 'Afternoon_Fluct_Max',
    'Evening_Inactive_Sum', 'Evening_Inactive_Max', 'Evening_Inactive_Mean',
    'Evening_Fluct_Sum', 'Evening_Fluct_Max',
    'Lag1_Mean_Inactive', 'Lag1_Mean_Fluct',
    'Rolling_Mean_Inactive', 'Rolling_Mean_Fluct',
    'Window_Inactive_Sum', 'Window_Inactive_Mean', 'Window_Fluct_Sum',
] + [f'Hour_{h}_Inactive' for h in range(1, 25)] \
  + [f'Hour_{h}_Fluctuation' for h in range(1, 25)]

# =============================================================
# STEP 3e — TIME-AWARE TRAIN / TEST SPLIT
# =============================================================
log("\n" + "=" * 62)
log("STEP 3e — TIME-AWARE TRAIN / TEST SPLIT")
log("=" * 62)

unique_dates = sorted(ml_df['Alarm_Raised_Date'].unique())
n_test_days  = 3
train_dates  = unique_dates[:-n_test_days]
test_dates   = unique_dates[-n_test_days:]

log(f"\n  Total dates: {len(unique_dates)}")
log(f"  Train dates ({len(train_dates)}): {[str(pd.Timestamp(d).date()) for d in train_dates]}")
log(f"  Test  dates ({len(test_dates)}):  {[str(pd.Timestamp(d).date()) for d in test_dates]}")

train_df = ml_df[ml_df['Alarm_Raised_Date'].isin(train_dates)].copy()
test_df  = ml_df[ml_df['Alarm_Raised_Date'].isin(test_dates)].copy()

log(f"\n  Train samples: {len(train_df):,}")
log(f"  Test  samples: {len(test_df):,}")
log(f"  Train/Test ratio: {len(train_df)/len(ml_df)*100:.1f}% / {len(test_df)/len(ml_df)*100:.1f}%")

# ── Fit scaler on TRAIN only, transform both ─────────────────
scaler = StandardScaler()
SCALE_COLS_PRESENT = [c for c in SCALE_COLS if c in FEATURE_COLS]

train_df[SCALE_COLS_PRESENT] = scaler.fit_transform(train_df[SCALE_COLS_PRESENT])
test_df[SCALE_COLS_PRESENT]  = scaler.transform(test_df[SCALE_COLS_PRESENT])

log(f"\n  StandardScaler fitted on train, applied to both")
log(f"  Scaled columns: {len(SCALE_COLS_PRESENT)}")

# ── Save scaler params ────────────────────────────────────────
scaler_meta = {
    'scaled_columns': SCALE_COLS_PRESENT,
    'mean_': scaler.mean_.tolist(),
    'scale_': scaler.scale_.tolist(),
}

# =============================================================
# SAVE ALL OUTPUTS
# =============================================================
log("\n" + "=" * 62)
log("SAVING OUTPUTS")
log("=" * 62)

# Full ML dataset (unscaled, for reference)
ml_df.to_csv(f'{OUTPUT_DIR}/ml_dataset.csv', index=False)
log(f"  ml_dataset.csv        — full dataset ({len(ml_df):,} rows)")

# Train / test feature and target splits
X_train = train_df[ID_COLS + FEATURE_COLS]
X_test  = test_df[ID_COLS + FEATURE_COLS]
y_train = train_df[ID_COLS + TARGET_COLS]
y_test  = test_df[ID_COLS + TARGET_COLS]

X_train.to_csv(f'{OUTPUT_DIR}/X_train.csv', index=False)
X_test.to_csv(f'{OUTPUT_DIR}/X_test.csv',   index=False)
y_train.to_csv(f'{OUTPUT_DIR}/y_train.csv', index=False)
y_test.to_csv(f'{OUTPUT_DIR}/y_test.csv',   index=False)

log(f"  X_train.csv           — {X_train.shape}")
log(f"  X_test.csv            — {X_test.shape}")
log(f"  y_train.csv           — {y_train.shape}")
log(f"  y_test.csv            — {y_test.shape}")

# Preprocessing metadata
meta = {
    'feature_columns':       FEATURE_COLS,
    'target_columns':        TARGET_COLS,
    'id_columns':            ID_COLS,
    'scale_columns':         SCALE_COLS_PRESENT,
    'train_dates':           [str(pd.Timestamp(d).date()) for d in train_dates],
    'test_dates':            [str(pd.Timestamp(d).date()) for d in test_dates],
    'constants': {
        'FLOOR_INACTIVE_SECONDS':   FLOOR_INACTIVE_SECONDS,
        'FLOOR_INACTIVE_MINUTES':   FLOOR_INACTIVE_MINUTES,
        'FLOOR_FLUCTUATION':        FLOOR_FLUCTUATION,
        'INACTIVE_CAP_PERCENTILE':  INACTIVE_CAP_PERCENTILE,
        'FLUCTUATION_CAP':          FLUCTUATION_CAP,
        'MIN_WINDOW_HOURS':         MIN_WINDOW_HOURS,
        'MAX_WINDOW_HOURS':         MAX_WINDOW_HOURS,
    },
    'scaler': scaler_meta,
}

with open(f'{OUTPUT_DIR}/preprocessing_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
log(f"  preprocessing_meta.json — scaler params, col lists, constants")

# Target label distribution summary
log(f"\n  === TARGET LABEL DISTRIBUTIONS ===")
for col in TARGET_COLS:
    vals = ml_df[col]
    log(f"  {col}:")
    log(f"    min={vals.min():.2f}  mean={vals.mean():.2f}  "
        f"median={vals.median():.2f}  max={vals.max():.2f}  std={vals.std():.2f}")

# Feature summary
log(f"\n  === FEATURE SUMMARY ===")
log(f"  Total features fed to models: {len(FEATURE_COLS)}")
feature_groups = {
    'Temporal (date/time)':     [c for c in FEATURE_COLS if c in [
        'DayOfWeek','IsWeekend','Month','DayOfMonth','DayOfYear',
        'DayOfWeek_sin','DayOfWeek_cos','Start_Hour_sin','Start_Hour_cos',
        'IsEvening','IsMorningPeak','IsOffPeak','Sensitivity_Factor']],
    'Hourly raw (48 cols)':     [c for c in FEATURE_COLS if 'Hour_' in c],
    'Block aggregations':       [c for c in FEATURE_COLS if any(
        b in c for b in ['Night_','Morning_','Afternoon_','Evening_'])],
    'Peak features':            [c for c in FEATURE_COLS if 'Peak' in c or
                                  'Active_Hour' in c or 'Concentration' in c],
    'Rolling/lag':              [c for c in FEATURE_COLS if 'Lag' in c or 'Rolling' in c],
    'Window context':           [c for c in FEATURE_COLS if 'Window' in c],
    'Vendor/Region encoded':    [c for c in FEATURE_COLS if
                                  c.startswith('Vendor_') or c.startswith('Region_')],
    'Log transforms':           [c for c in FEATURE_COLS if 'Log_' in c],
}
for group, cols in feature_groups.items():
    log(f"    {group}: {len(cols)} features")

# Save report
with open(f'{OUTPUT_DIR}/preprocessing_report.txt', 'w') as f:
    f.write('\n'.join(REPORT_LINES))
log(f"\n  preprocessing_report.txt — full decisions log")

log("\n" + "=" * 62)
log("PREPROCESSING COMPLETE")
log("=" * 62)
log(f"\n  All outputs saved to: {OUTPUT_DIR}/")
log(f"\n  Next step: Build 4 ML models using X_train, y_train, X_test, y_test")

TELECOM ALARM ML PREPROCESSING PIPELINE

[LOAD] Reading anonymized dataset...
  Loaded: 14054 rows × 56 cols
  Dates:  2024-06-08 → 2024-06-24
  Vendors: ['Vendor_X', 'Vendor_Y', 'Vendor_Z']

STEP 3a — DATA CLEANING

  [3a-1] REG_OTHER (anomalous hash cells): 8 rows → removed
  [3a-2] Inactive time capped at P99 = 35946s (599.1 min)
         Rows affected: 141
  [3a-3] Fluctuation capped at 7 (P99 value)
         Rows affected: 140
  [3a-4] Totals recalculated from hourly values after capping

  Rows after cleaning: 14046 (removed 8)

STEP 3b — FEATURE ENGINEERING

  [3b-2] Temporal features:
    DayOfWeek, IsWeekend, Month, DayOfMonth, DayOfYear
    DayOfWeek_sin, DayOfWeek_cos (cyclical encoding)

  [3b-3] Hourly aggregation features:
    4 time blocks × 6 features = 24 block features
    Peak hour features: H9 inactive, H24 inactive, H10 fluctuation
    Total_Active_Hours, Active_Hour_Ratio
    Inactive_Concentration, Fluct_Concentration (peak-hour dominance ratio)
    Log_Total_Ina

UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 262: character maps to <undefined>